# College basketball: matchup matrix

Neutral-site win probabilities between every pair of current teams, e.g. for
March Madness. Needs Kaggle credentials (`~/.kaggle/kaggle.json`) and a
trained model:

```sh
sportsml cbb download && sportsml cbb process && sportsml cbb fit gnn
```

Swap `cbb` for `wcbb` for the women's game.

In [ ]:
import pandas as pd

from sportsml.core.predictor import load_predictor
from sportsml.inference import all_matchups, long_to_wide

games = pd.read_csv("../data/cbb/games.csv")
predictor = load_predictor("../models/cbb/gnn")
predictor.update_state(games)  # team state as of the latest games

In [ ]:
pairs = predictor.predict(all_matchups(predictor))  # location 0 = neutral
names = games.drop_duplicates("TeamID").set_index("TeamID")["Team"].to_dict()
ratings = pairs.groupby("TeamID")[["pred_margin", "win_prob"]].mean()
ratings.rename(index=names).sort_values("pred_margin", ascending=False).head(25)

In [ ]:
long_to_wide(pairs, team="TeamID", opp="TeamID_OPP", team_name_map=names).round(2)

Bracket simulation: `sportsml.cbb.data.bracket.Bracket` takes Kaggle's
`MNCAATourneySlots.csv` / `MNCAATourneySeeds.csv`, and `ProbabilityPredictor`
takes `long_to_wide(pairs, team="TeamID", opp="TeamID_OPP", sorted=False)`.